# Logging

> Generated from [`07-Logging.py`](07-Logging.py) by `tools/build_notebooks.py` - edit the `.py` (or its `.md` notes), not this notebook. Changes made here are overwritten on the next build.

## Quick notes

### What this covers
The `logging` module — the real tool for reporting what a program is doing, with severity levels, timestamps, and configurable output, instead of scattering `print()` everywhere.

### Key points
- **Levels, lowest to highest**: `DEBUG < INFO < WARNING < ERROR < CRITICAL`. `basicConfig(level=X)` shows `X` and everything more severe.
- **`logging.basicConfig(level=..., format=..., datefmt=...)`** — quick one-time setup for the root logger.
- **`logging.getLogger(__name__)`** — the recommended pattern: one logger per module. `__name__` is the module's dotted path, so it auto-slots into a hierarchy.
- **`logger.exception("msg")`** — call inside an `except` block to log the message plus the full traceback.
- **`FileHandler`** — write a logger's output to a file; `.setFormatter()` controls its format independently of other handlers.
- **`logger.propagate = False`** — stop a logger's messages from also bubbling up to its parent/root handlers (e.g. avoid double-printing).
- **Logger hierarchy** — loggers form a tree from dotted names (`"app"` is parent of `"app.module"`); a child's messages propagate up to every ancestor's handlers unless `propagate=False`.
- **`RotatingFileHandler`** (`logging.handlers`) — caps a log file at `maxBytes`, keeps `backupCount` rotated old copies once it's full.
- **`logging.config.dictConfig({...})`** — declarative setup for multiple loggers/handlers/formatters at once (e.g. loaded from JSON/YAML), instead of many imperative calls.
- **`extra={...}`** — attach custom fields (e.g. `user_id`) to a log record, usable in the format string as `%(user_id)s`.

### Gotchas (the part worth revising twice)
- Use `logging.getLogger(__name__)` per module, **not** one global/root logger — it lets you tune verbosity or redirect output per subsystem later, and log records carry their true origin.
- A child logger's messages propagate **up** the hierarchy by default — if you attach a handler to a child and also see duplicate output from the root, that's why; set `propagate = False` to stop it.
- `print()` is fine for quick one-off checks or CLI output meant for the end user; `logging` is for real applications where severity, timestamps, and per-module control matter.
- `RotatingFileHandler` rotates by file size (`maxBytes`), not by time — check `TimedRotatingFileHandler` if you need day/hour-based rotation instead.

### When it matters
Any application beyond a quick script — debugging production issues, tuning verbosity per subsystem without code changes, or writing structured logs to files/services.

### One-line recap example
```python
import logging
logger = logging.getLogger(__name__)          # one logger per module
logger.info("processing started")
```

In [ ]:
import logging

## 01. Logging - Introduction

- print() is fine for quick checks; logging is the real tool for apps -
  severity levels, timestamps, output to file/console, filterable.

## 02. Basic Configuration and Logging Levels

In [ ]:
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%H:%M:%S",
)

print("\n# 02. Basic Configuration and Logging Levels")
logging.debug("Debug message - detailed diagnostic info")
logging.info("Info message - confirms things work as expected")
logging.warning("Warning message - something unexpected happened")
logging.error("Error message - a real problem occurred")
logging.critical("Critical message - program may be unable to continue")

## 03. Level Order (lowest to highest severity)

## 04. Named Loggers (per-module, the recommended pattern)

In [ ]:
logger = logging.getLogger(__name__)  # __name__ = module's own name

print("\n# 04. Named Loggers")
logger.info("Message from a named logger: %s", __name__)

## 05. Logging Exceptions with Traceback

In [ ]:
try:
    1 / 0
except ZeroDivisionError:
    logger.exception("Caught an exception")  # includes full traceback

## 06. Logging to a File

In [ ]:
file_logger = logging.getLogger("file_demo")
file_handler = logging.FileHandler("temp_app.log")
file_handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
file_logger.addHandler(file_handler)
file_logger.setLevel(logging.INFO)
file_logger.propagate = False  # don't also print via the root logger's console handler

file_logger.info("This line goes to temp_app.log")

with open("temp_app.log") as f:
    print("file content:", f.read().strip())

## 07. Cleanup

In [ ]:
import os
file_handler.close()
os.remove("temp_app.log")
print("\n# 07. Cleanup")
print("temp_app.log removed")

## 08. print() vs logging - When to Use Which

## 09. Logger Hierarchy and Propagation

- Loggers form a tree based on dotted names: "app" is the parent of
  "app.module", which is the parent of "app.module.sub".
- A message logged on a child propagates UP to every ancestor's handlers
  (unless propagate=False, as used for file_logger above).

In [ ]:
parent_logger = logging.getLogger("app")
child_logger = logging.getLogger("app.module")
print("child's parent:", child_logger.parent.name)
print("child propagates to parent:", child_logger.propagate)
child_logger.warning("this bubbles up through 'app' to the root logger's handlers")

## 10. getLogger(__name__) - Why It Matters at Scale

- __name__ is the dotted module path (e.g. "myapp.utils.parser"), so
  every module's logger automatically slots into the right place in the
  hierarchy. You can then silence or redirect one subsystem
  (logging.getLogger("myapp.utils").setLevel(logging.WARNING)) without
  touching every print statement or a single global logger's config.

## 11. RotatingFileHandler - Log Rotation by Size

In [ ]:
from logging.handlers import RotatingFileHandler

print("\n# 11. RotatingFileHandler")
rotating_logger = logging.getLogger("rotating_demo")
rotating_logger.propagate = False
rotating_handler = RotatingFileHandler(
    "temp_rotating.log", maxBytes=200, backupCount=2
)
rotating_logger.addHandler(rotating_handler)
rotating_logger.setLevel(logging.INFO)
for i in range(20):
    rotating_logger.info(f"log line number {i} - padded to trigger rotation soon")

import glob
rotating_handler.close()
rotated_files = sorted(glob.glob("temp_rotating.log*"))
print("files created by rotation:", rotated_files)
for f in rotated_files:
    os.remove(f)

## 12. logging.config.dictConfig - Structured Configuration

- basicConfig() is fine for one quick setup; dictConfig() lets you define
  multiple loggers/handlers/formatters declaratively (e.g. loaded from a
  YAML/JSON config file) instead of imperative setup calls.

In [ ]:
import logging.config

dict_config = {
    "version": 1,
    "disable_existing_loggers": False,
    "formatters": {
        "simple": {"format": "%(name)s - %(levelname)s - %(message)s"},
    },
    "handlers": {
        "console": {"class": "logging.StreamHandler", "formatter": "simple"},
    },
    "loggers": {
        "dictconfig_demo": {"handlers": ["console"], "level": "INFO", "propagate": False},
    },
}
logging.config.dictConfig(dict_config)
print("\n# 12. logging.config.dictConfig")
logging.getLogger("dictconfig_demo").info("configured entirely via a dict")

## 13. extra= - Adding Contextual Info to Log Records

In [ ]:
context_logger = logging.getLogger("context_demo")
context_logger.propagate = False
context_handler = logging.StreamHandler()
context_handler.setFormatter(logging.Formatter("%(levelname)s - user=%(user_id)s - %(message)s"))
context_logger.addHandler(context_handler)
context_logger.setLevel(logging.INFO)
context_logger.info("processing request", extra={"user_id": 42})